<a href="https://colab.research.google.com/github/niikun/ezkl/blob/main/notebooks/01_onnx.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01. ONNX エクスポートと内部構造

`simple_demo.ipynb` の **Step 1（ONNX エクスポート）** を深く理解するためのノートブックです。

**前提ファイル**: なし（このノートブック内でモデルを訓練します）

In [1]:
try:
    import google.colab, subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "ezkl", "onnx", "torch", "torchvision"])
except ImportError:
    pass

In [2]:
from torch import nn

class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 2, kernel_size=5, stride=2)
        self.conv2 = nn.Conv2d(2, 3, kernel_size=5, stride=2)
        self.relu  = nn.ReLU()
        self.d1    = nn.Linear(48, 48)
        self.d2    = nn.Linear(48, 10)

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = x.view(-1, 48)
        x = self.relu(self.d1(x))
        return self.d2(x)

## モデルを訓練して ONNX にエクスポートする

まず簡単に訓練して、ONNX ファイルを作ります。

In [39]:
import torch, torchvision, json, os
from torch import nn
from torch.utils.tensorboard import SummaryWriter

writer = SummaryWriter('logs')

transform = torchvision.transforms.Compose([
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize((0.5,), (0.5,))
])
dataset    = torchvision.datasets.FashionMNIST('./data', train=True, download=True, transform=transform)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=64, shuffle=True)

model     = MyModel()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(3):
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0
    for inputs, labels in dataloader:
        optimizer.zero_grad()
        y = model(inputs)
        loss = criterion(y, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0) # Accumulate batch loss, weighted by batch size

        y_pred = torch.argmax(y, dim=1)
        correct_predictions += (y_pred == labels).sum().item()
        total_samples += labels.size(0)

    epoch_loss = running_loss / total_samples
    epoch_accuracy = correct_predictions / total_samples

    writer.add_scalar('Accuracy/train', epoch_accuracy, epoch)
    writer.add_scalar('Loss/train', epoch_loss, epoch)

    print(f"Epoch {epoch+1} 完了, Loss: {epoch_loss:.4f}, Accuracy: {epoch_accuracy:.4f}")
writer.close()

Epoch 1 完了, Loss: 0.8255, Accuracy: 0.6987
Epoch 2 完了, Loss: 0.5668, Accuracy: 0.7880
Epoch 3 完了, Loss: 0.5146, Accuracy: 0.8094


In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs

In [4]:
model_path = 'network.onnx'
data_path  = 'input.json'
shape      = [1, 1, 28, 28]

model.eval()
x = torch.randn(1, *shape[1:], requires_grad=True)

torch.onnx.export(
    model, x, model_path,
    export_params=True, opset_version=10, do_constant_folding=True,
    input_names=['input'], output_names=['output'], dynamo=False
)
json.dump({"input_data": [x.detach().numpy().reshape(-1).tolist()]}, open(data_path, 'w'))
print(f"エクスポート完了: {model_path}  ({os.path.getsize(model_path):,} bytes)")

エクスポート完了: network.onnx  (13,420 bytes)


/tmp/ipykernel_6690/1806316957.py:8: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


`torch.randn(...)`：ランダムな値が入った $1 \times 1 \times 28 \times 28$ のダミーの画像データ（x）を作っています。  
ONNXへエクスポートする際、PyTorchは実際にデータを1回モデルに流してみて、計算の流れを追跡（トレース）する必要があるため、このダミーデータが必要になります。

## ONNX とは何か

**ONNX（Open Neural Network Exchange）** = ニューラルネットワークを「どのフレームワークでも読める形式」で保存する規格。

EZKL が ONNX を使う理由：
```
PyTorch モデル
      ↓  torch.onnx.export
  network.onnx   ← EZKL が読み込む
      ↓  ezkl.compile_circuit
    ZK 回路
```

ONNX の中身は **有向非巡回グラフ（DAG）** で、ノード（操作）とエッジ（データの流れ）で構成されます。

## グラフ全体の確認

In [5]:
import onnx

onnx_model = onnx.load(model_path)
graph      = onnx_model.graph

print(f"ONNX バージョン  : {onnx_model.ir_version}")
print(f"プロデューサー   : {onnx_model.producer_name} {onnx_model.producer_version}")
print(f"ノード総数       : {len(graph.node)}")
print(f"重み（initializer）: {len(graph.initializer)}")

ONNX バージョン  : 5
プロデューサー   : pytorch 2.11.0
ノード総数       : 9
重み（initializer）: 8


In [9]:
onnx_model

ModelProto(ir_version=5, opset_import={'': 10}, producer_name='pytorch', producer_version='2.11.0', graph=GraphProto('main_graph', input=<1 inputs>, output=<1 outputs>, initializer=<8 initializers>, node=<9 nodes>))

## 入力・出力の shape を確認する

In [6]:
print("=== グラフ入力 ===")
for inp in graph.input:
    shape = [d.dim_value for d in inp.type.tensor_type.shape.dim]
    print(f"  {inp.name}: shape={shape}")

print("\n=== グラフ出力 ===")
for out in graph.output:
    shape = [d.dim_value for d in out.type.tensor_type.shape.dim]
    print(f"  {out.name}: shape={shape}")

=== グラフ入力 ===
  input: shape=[1, 1, 28, 28]

=== グラフ出力 ===
  output: shape=[1, 10]


## 全ノード（計算ステップ）を順番に確認する

各ノードは「op_type（何をするか）」「inputs（何を受け取るか）」「outputs（何を返すか）」で構成されます。

In [7]:
for i, node in enumerate(graph.node):
    print(f"[{i:02d}] {node.op_type:10s} | 入力: {list(node.input)} | 出力: {list(node.output)}")

[00] Conv       | 入力: ['input', 'conv1.weight', 'conv1.bias'] | 出力: ['/conv1/Conv_output_0']
[01] Relu       | 入力: ['/conv1/Conv_output_0'] | 出力: ['/relu/Relu_output_0']
[02] Conv       | 入力: ['/relu/Relu_output_0', 'conv2.weight', 'conv2.bias'] | 出力: ['/conv2/Conv_output_0']
[03] Relu       | 入力: ['/conv2/Conv_output_0'] | 出力: ['/relu_1/Relu_output_0']
[04] Constant   | 入力: [] | 出力: ['/Constant_output_0']
[05] Reshape    | 入力: ['/relu_1/Relu_output_0', '/Constant_output_0'] | 出力: ['/Reshape_output_0']
[06] Gemm       | 入力: ['/Reshape_output_0', 'd1.weight', 'd1.bias'] | 出力: ['/d1/Gemm_output_0']
[07] Relu       | 入力: ['/d1/Gemm_output_0'] | 出力: ['/relu_2/Relu_output_0']
[08] Gemm       | 入力: ['/relu_2/Relu_output_0', 'd2.weight', 'd2.bias'] | 出力: ['output']


## op_type の意味

| op_type | PyTorch の対応 | 内容 |
|---|---|---|
| `Conv` | `nn.Conv2d` | 畳み込み（重みとの内積） |
| `Relu` | `nn.ReLU` | 0 以下を 0 にする |
| `Constant` | 定数 | reshape に使うサイズ定数 |
| `Reshape` | `.view()` | テンソルの形状変換（Flatten） |
| `Gemm` | `nn.Linear` | 行列積 + バイアス |

`Gemm` = General Matrix Multiply。`nn.Linear` は内部で `output = input @ weight.T + bias` を計算しており、これが Gemm に対応します。

## モデルの重み（initializer）を確認する

`graph.initializer` に Conv の kernel・bias、Linear の weight・bias が格納されています。

In [41]:
graph.initializer[0].raw_data

b"b\xa7\x86>~{\xa1>xy\xfa>\x0f\xe4k>\x11\xd2\xc6\xbb\xa2\xb5\x1e\xbd%[\x1e>\xe8\x07\xea>\xef'\xca>q`\xb6=\xbd\x01&\xbeF\x9b\x82>R\xd1]>\xd1\x1b\x19>Q\xc5\x18\xbd\xeb\xf7\xe3=l%\xa8>\x85\xe5\xc8>\xe6\xeb\t>\xc4\x9b\xc3>H\xcd\x05>\x1b\xf5\xb3>\xfb\xc2\xad>Q\xe0\x00>\x11\xc3\xbc>1\xd3\xce\xbd\xd4\xe9\x1c\xbf\x0c)&\xbfLd\xac\xbe\x05\x06\xb5\xbd\xd6^\x88\xbeb\xa0\xd1\xbe\r\xab&\xbf\x90\xc9\x92\xbe`L\xae\xbeb\x1b^\xb9\xdd\x9d\r\xbfK\xd3\xa3\xbe\xe3B\xb1\xbcE\x1d\t\xbd\x11\xc9\xc5=\x97\xd4\xcd\xbe\x19\x06T\xbe\x06C3\xbew\x01\xd5<\xff\xbdU\xbc\x95\xa9%\xbe\xe6\xfd\x88\xbe\xfa\xdb\x87\xbd\x07\xe0!\xbe"

In [8]:
total = 0
for init in graph.initializer:
    shape = list(init.dims)
    n = 1
    for d in shape: n *= d
    total += n
    print(f"  {init.name:35s}  shape={str(shape):20s}  {n:6,} params")
print(f"\n総パラメータ数: {total:,}")

  conv1.weight                         shape=[2, 1, 5, 5]              50 params
  conv1.bias                           shape=[2]                        2 params
  conv2.weight                         shape=[3, 2, 5, 5]             150 params
  conv2.bias                           shape=[3]                        3 params
  d1.weight                            shape=[48, 48]               2,304 params
  d1.bias                              shape=[48]                      48 params
  d2.weight                            shape=[10, 48]                 480 params
  d2.bias                              shape=[10]                      10 params

総パラメータ数: 3,047


## shape がどう変わるか追う

入力から出力まで、各操作で shape がどう変わるかを確認します。

In [11]:
import torch

model.eval()
hooks = []
shapes = []

def hook(name):
    def fn(module, inp, out):
        shapes.append((name, tuple(inp[0].shape), tuple(out.shape)))
    return fn

for name, m in model.named_modules():
    if isinstance(m, (torch.nn.Conv2d, torch.nn.ReLU, torch.nn.Linear)):
        hooks.append(m.register_forward_hook(hook(name)))

with torch.no_grad():
    model(torch.randn(1, 1, 28, 28))

for h in hooks:
    h.remove()

print(f"{'レイヤー':10s}  {'入力 shape':25s}  {'出力 shape'}")
print("-" * 60)
for name, inp, out in shapes:
    print(f"{name:10s}  {str(inp):25s}  {str(out)}")

レイヤー        入力 shape                   出力 shape
------------------------------------------------------------
conv1       (1, 1, 28, 28)             (1, 2, 12, 12)
relu        (1, 2, 12, 12)             (1, 2, 12, 12)
conv2       (1, 2, 12, 12)             (1, 3, 4, 4)
relu        (1, 3, 4, 4)               (1, 3, 4, 4)
d1          (1, 48)                    (1, 48)
relu        (1, 48)                    (1, 48)
d2          (1, 48)                    (1, 10)


## ZK 回路との接続

**各 ONNX ノード = ZK 制約の塊** です。

```
[Conv]    → 掛け算・足し算の制約が大量に発生
[Relu]    → 出力 >= 0 の範囲証明制約
[Gemm]    → 行列積の乗算制約
[Reshape] → データ移動のみ（制約コストは低い）
```

パラメータ数が多いほど制約数が増え、証明生成に時間・メモリがかかります。

**Netron で可視化**: https://netron.app に `network.onnx` をドロップするとグラフを見られます。

---
次: `02_settings.ipynb` で visibility（公開・秘密・固定）を設定します。